# UNVERIFIED / INCOMPLETE NOTEBOOK

**未検証・途中までの参考Notebookです。** 全セルの実行成功は確認していません。

**Stopping reason:** Notebook authored and statically validated (CPU scope: authors' loadMIAPPE workflow from Data in Brief 62:112004 Fig. 2 on study TTADDA_NARO_2021). Two full Colab runs and six probes failed because data.4tu.nl was down: download endpoints returned 502 or stalled with no first byte for ~2 h (landing pages intermittently 200); the whole site was unreachable by the end. The MIAPPE xlsx and measurement CSVs are verified published (4TU items 20ef3f20/f2307c47 v1, xlsx 2,328,999 B MD5 1e935e229dc27c74efc9ff9a6f7da45b) but bytes could not be retrieved. Notebook retained in deliverables/ as unverified reference; rerun when 4TU recovers.

Attempt status: failed. No manual login validation was performed. This draft may require your own authorized access, fail partway, or contain incomplete code. This attempt does not establish that the paper cannot be reproduced.

Paper: TTADDA-UAV: A multi-season RGB and multispectral UAV dataset of potato fields collected in Japan and the Netherlands

[PhenoPaper record](https://phenopaper.smartbreed-plant-phenotyping-platform.com/papers/p-89e0e604d88be6ac51289b5224ee3bd0)


---
**STATUS: NOT EXECUTED — VALIDATION FAILED (2026-10-05).**
このノートブックは実行検証されていません / This notebook has not been executed successfully.

The notebook is complete and statically validated, but its data source **data.4tu.nl (4TU.ResearchData)**
suffered a full outage on 2026-10-05 (download endpoints returned 502 / connection failures from both
DGX and Colab; landing pages intermittently 200). The MIAPPE workbook and the TTADDA_NARO_2021
measurement CSVs are published and verified (item metadata, sizes, MD5), but the bytes could not be
retrieved during the validation window. Re-run this notebook when 4TU is reachable.

検証時点で 4TU.ResearchData が障害中のため、データ取得に失敗しました。公開データ自体は確認済みです
（項目メタデータ・MD5 検証値はノートブック内に記載）。4TU が復旧後に Run all で再実行してください。
---
# TTADDA-UAV: loading the MIAPPE metadata and extracting ground coverage & yield (TTADDA_NARO_2021)

Reproduction of the data workflow shown in **Fig. 2** of:

> van Marrewijk B.M., Njane S.N., Tsuda S., van Culemborg M., Polder G., Katayama K., van Daalen T., van de Zedde R.
> *TTADDA-UAV: A multi-season RGB and multispectral UAV dataset of potato fields collected in Japan and the Netherlands.*
> **Data in Brief** 62:112004 (2025). doi:[10.1016/j.dib.2025.112004](https://doi.org/10.1016/j.dib.2025.112004) — open access via [PMC12433462](https://pmc.ncbi.nlm.nih.gov/articles/PMC12433462/)

**Scope (one study, one example):** this notebook uses the authors' own loader
(`NPEC-NL/MIAPPE_TTADDA_dataset` @ commit `2fd6e19`) to load the public MIAPPE workbook
(4TU item, CC BY-NC-SA 4.0), list the observation units of study `TTADDA_NARO_2021`,
extract the **ground coverage** time series of its plots and the **harvest yield**
(`tubwght_total_kgm-2`) per plot, mirroring the paper's Fig. 2 loader demonstration.

**Not in scope:** the ~550 GB of orthomosaic / DSM / multispectral imagery (only small
metadata and measurement CSV files are downloaded). This is a data-descriptor paper: there
are no models, no weights and no accuracy claims to reproduce.

**実行環境 (runtime):** CPU のみで動作します（pandas + matplotlib の表・グラフ処理のみ）。
Colab で「ランタイム」→「ランタイムのタイプを変更」は不要です。**Run all** で先頭から実行してください。


## 0. How this notebook gets its data (provenance)

| Artifact | Source | Verification |
| --- | --- | --- |
| Author loader code | `raw.githubusercontent.com/NPEC-NL/MIAPPE_TTADDA_dataset @ 2fd6e1902e618619a7d236c9007b9e4b9cd6aac7` | pinned commit, fetched inside this notebook |
| MIAPPE workbook | 4TU item `20ef3f20-ec7b-4236-b13b-e4f7d54a34bf` v1, file `MIAPPE_Minimal_Spreadsheet_Template_TTADDAv4.xlsx` | expected size 2,328,999 bytes, MD5 `1e935e229dc27c74efc9ff9a6f7da45b` (from the published item page) |
| Measurement CSVs | 4TU item `f2307c47-9a1a-474a-a0d9-e09ee1b7512c` v1 (TTADDA_NARO_2021), **selective extraction only** | we list the remote ZIP's central directory and download only the small CSVs referenced by the workbook |

The full TTADDA_NARO_2021 item is ~12.6 GB because it contains all drone orthomosaics.
To keep the demo lightweight we never download that archive in full: the ZIP central
directory is read with HTTP `Range` requests (`remotezip`), and only the measurement CSVs
needed for ground coverage and yield are fetched. **データは 4TU 公開リポジトリから、
このノートブック内（Colab VM）でのみダウンロードされます。**

Expected end-to-end time: ~5 minutes (est.), downloads well under 100 MB (est.).


## 1. Setup — install the small set of dependencies the authors' loader needs

The authors' README lists `omegaconf, pandas, geopandas, openpyxl` (Python 3.11).
On Colab, `pandas`, `matplotlib` and `openpyxl` are preinstalled; we add `natsort`,
`geopandas` (imported by `scripts/data_loader.py`), `omegaconf` (imported by `config.py`)
and `remotezip` for selective ranged downloads.

Expected result: pip finishes without errors and prints the installed versions.


In [ ]:
%pip install -q natsort geopandas omegaconf remotezip

import importlib
for pkg in ["pandas", "openpyxl", "natsort", "geopandas", "remotezip", "matplotlib"]:
    m = importlib.import_module(pkg)
    print(f"{pkg:12s} {getattr(m, '__version__', '?')}")


## 2. Fetch the authors' loader code (pinned revision)

We download the exact files of the authors' repository at commit
`2fd6e1902e618619a7d236c9007b9e4b9cd6aac7` — the commit that accompanied the paper —
and place them on the Colab VM so the original `loadMIAPPE` class can be imported unchanged.
No algorithm is re-implemented: the authors' own code performs the loading.

Expected result: the files are saved under `/content/repo/` and listed.


In [ ]:
import urllib.request
from pathlib import Path

REPO_URL = "https://raw.githubusercontent.com/NPEC-NL/MIAPPE_TTADDA_dataset"
PINNED_COMMIT = "2fd6e1902e618619a7d236c9007b9e4b9cd6aac7"
FILES = [
    "scripts/data_loader.py",
    "config.py",
    "config.yaml",
    "main.py",
    "readme.md",
]

REPO_DIR = Path("/content/repo")
for rel in FILES:
    dest = REPO_DIR / rel
    dest.parent.mkdir(parents=True, exist_ok=True)
    url = f"{REPO_URL}/{PINNED_COMMIT}/{rel}"
    urllib.request.urlretrieve(url, dest)
    print(f"{rel:28s} {dest.stat().st_size:>7d} bytes")

# touch __init__.py so scripts/ is importable as a package
(REPO_DIR / "scripts" / "__init__.py").write_text("")


## 3. Download the MIAPPE workbook from 4TU and verify its checksum

The workbook is the single entry point the authors' loader uses: seven MIAPPE sheets
(Investigation, Study, Observation Unit, Biological Material, Data file, Sensor,
Observed Variable, Events). We fetch the published file directly with its per-file
download URL and **verify size and MD5** against the values published on the 4TU item page
before using it.

Expected result: `MD5 OK / size OK` and the list of sheet names (seven structured sheets).


In [ ]:
import hashlib
import io
import time
import zipfile
import requests

XLSX_NAME = "MIAPPE_Minimal_Spreadsheet_Template_TTADDAv4.xlsx"
# documented contentUrl of the metadata item (4TU item 20ef3f20-…, v1; from the item's
# JSON-LD and the authors' config.yaml). The download is a ZIP wrapping the workbook.
ITEM_ZIP_URL_X = "https://data.4tu.nl/ndownloader/items/20ef3f20-ec7b-4236-b13b-e4f7d54a34bf/versions/1"
XLSX_SIZE = 2_328_999          # published on the 4TU item page (2025-09-03, v1)
XLSX_MD5 = "1e935e229dc27c74efc9ff9a6f7da45b"

DATA_DIR = Path("/content/TTADDA_download")
DATA_DIR.mkdir(parents=True, exist_ok=True)


def fetch(url, dest, timeout=(30, 180), retries=8, wait_s=45):
    """Stream a URL to disk with finite timeouts and patient bounded retries.

    data.4tu.nl applies rate limits (Crawl-delay 20) and occasionally returns 502s or
    stalls; every failure is retried with a fixed pause instead of failing fast.
    """
    last = None
    for attempt in range(retries):
        try:
            with requests.get(url, stream=True, timeout=timeout, allow_redirects=True) as r:
                r.raise_for_status()
                with open(dest, "wb") as f:
                    for chunk in r.iter_content(chunk_size=1 << 16):
                        if chunk:
                            f.write(chunk)
            return dest
        except Exception as e:
            last = e
            print(f"attempt {attempt + 1}/{retries} failed: {type(e).__name__}: {e}")
            if attempt + 1 < retries:
                time.sleep(wait_s)
    raise RuntimeError(f"download failed: {url}") from last


xlsx_path = DATA_DIR / XLSX_NAME
item_zip = DATA_DIR / "TTADDA_metadata.zip"
fetch(ITEM_ZIP_URL_X, item_zip)
with zipfile.ZipFile(item_zip) as zf:
    print("members:", zf.namelist())
    inner = [n for n in zf.namelist() if n.endswith(".xlsx")]
    assert inner, "no xlsx member found"
    xlsx_path.write_bytes(zf.read(inner[0]))

raw = xlsx_path.read_bytes()
print(f"downloaded {len(raw)} bytes")
md5 = hashlib.md5(raw).hexdigest()
assert md5 == XLSX_MD5, f"MD5 mismatch: {md5}"
assert len(raw) == XLSX_SIZE, f"size mismatch: {len(raw)}"
print("MD5 OK, size OK")

import pandas as pd
sheets = pd.read_excel(xlsx_path, sheet_name=None)
print("sheets:", list(sheets.keys()))


## 4. Inspect which measurement files the workbook links for ground coverage and yield

Before downloading anything from the big study archive we ask the workbook itself which
files it needs: the `Observed Variable` sheet maps `variableId` → `dataId`, and the
`Data file` sheet maps `dataId` → `dataFileLink` (a path *relative to the dataset root*).
We list the distinct CSV links used by study `TTADDA_NARO_2021` for `ground_coverage` and
`tubwght_total_kgm-2` (yield).

Expected result: a handful of `.csv` paths (small files), plus any other variable types
the study uses (shown for context; we will not download imagery).


In [ ]:
df_obs = sheets["Observed Variable"]
df_files = sheets["Data file"]

study = "TTADDA_NARO_2021"

links = {}
for vid in ["ground_coverage", "tubwght_total_kgm-2"]:
    rows = df_obs[(df_obs["studyId"] == study) & (df_obs["variableId"] == vid)]
    for did in rows["dataId"].unique():
        link = df_files[df_files["dataId"] == did]["dataFileLink"].values[0]
        links.setdefault(vid, set()).add(link)

all_links = sorted({l for s in links.values() for l in s})
print("CSV files needed for this demo:")
for l in all_links:
    print("  ", l)

# context: which file types does the whole study reference?
kind = df_files["dataFileLink"].dropna().map(lambda s: s.rsplit(".", 1)[-1].lower())
print("\nfile extensions referenced by TTADDA_NARO_2021 Data file sheet:")
print(kind.value_counts().to_string())


## 5. Locate the measurement archive of study `TTADDA_NARO_2021`

The `TTADDA_NARO_2021` 4TU item contains a readme, the MIAPPE workbook and a large ZIP
with all drone imagery and measurements (~12.6 GB total). We resolve the item's per-file
download UUIDs from its public landing page and identify the measurement archive by its
size (via a 1-byte ranged request) — **without downloading any payload on the host of
this notebook**.

Expected result: the printed file UUIDs with their sizes; the ~12.6 GB archive identified.


In [ ]:
import io
import re
import requests


def fetch_html(url, retries=5, wait_s=45):
    """GET a page with patient bounded retries (4TU rate limits and 502s are transient)."""
    for attempt in range(retries):
        try:
            r = requests.get(url, timeout=(30, 180),
                             headers={"User-Agent": "colab-repro-notebook/1.0"})
            r.raise_for_status()
            return r.text
        except Exception as e:
            print(f"attempt {attempt + 1}/{retries} failed: {type(e).__name__}: {e}")
            if attempt + 1 < retries:
                time.sleep(wait_s)
    raise RuntimeError(f"page fetch failed: {url}")


NARO_ITEM_UUID = "f2307c47-9a1a-474a-a0d9-e09ee1b7512c"
NARO_ITEM_URL = f"https://data.4tu.nl/datasets/{NARO_ITEM_UUID}"
NDOWNLOADER_FILES = "https://data.4tu.nl/ndownloader/files/"
ITEM_ZIP_URL = f"https://data.4tu.nl/ndownloader/items/{NARO_ITEM_UUID}/versions/1"
PROJECT_DIR = Path("/content/data/TTADDA_UAV_01")   # matches config.yaml project_dir layout
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# 1) resolve the per-file UUIDs of the study item from its public landing page
page = fetch_html(NARO_ITEM_URL)
file_ids = sorted(set(re.findall(rf"/file/{NARO_ITEM_UUID}/([0-9a-f-]{{36}})", page)))
print(f"{len(file_ids)} files in the TTADDA_NARO_2021 item:", file_ids)

# 2) test whether the per-file download route works and record published sizes
file_urls = {}
for fid in file_ids:
    url = NDOWNLOADER_FILES + fid
    try:
        with requests.get(url, headers={"Range": "bytes=0-0"}, stream=True,
                          timeout=(30, 120), allow_redirects=True) as r:
            cr = r.headers.get("Content-Range", "")
            size = int(cr.rsplit("/", 1)[-1]) if "/" in cr else -1
            print(f"  {fid}: HTTP {r.status_code} size={size}")
            if r.status_code == 200 or r.status_code == 206:
                file_urls[fid] = url
    except Exception as e:
        print(f"  {fid}: {type(e).__name__}: {e}")
print("working per-file routes:", len(file_urls))


### Selective extraction with HTTP `Range` requests

The ~12.6 GB archive stays on the 4TU server. If 4TU exposes the large measurement ZIP
as its own file download, `remotezip` reads its central directory via `Range` requests
and we fetch **only** the small measurement CSVs. Otherwise we apply the same ranged
technique *inside* the item download ZIP (4TU wraps the item's files in a ZIP; a
zip-of-zip with stored members can be read positionally without downloading it).

Expected result: the needed `.csv` files appear under `/content/data/TTADDA_UAV_01/…`
with their real sizes printed, and the total downloaded volume stays far below 1 GB.


In [ ]:
import struct
import zipfile
from remotezip import RemoteZip


class HTTPRangeWindow(io.RawIOBase):
    """Read-only seekable window [base, base+size) of a remote URL via Range requests."""

    def __init__(self, url, base, size, timeout=(30, 300)):
        self.url, self.base, self.size, self.timeout = url, base, size, timeout
        self.pos = 0

    def seekable(self):
        return True

    def seek(self, offset, whence=0):
        self.pos = offset if whence == 0 else self.pos + offset if whence == 1 else self.size + offset
        return self.pos

    def tell(self):
        return self.pos

    def readable(self):
        return True

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        headers = {"Range": f"bytes={self.base + self.pos}-{self.base + self.pos + n - 1}"}
        with requests.get(self.url, headers=headers, timeout=self.timeout) as r:
            r.raise_for_status()
            data = r.content
        self.pos += len(data)
        return data


def extract_small_members(zf, extract_root):
    """Extract only csv/xlsx/md members of an open ZipFile; returns count."""
    n = 0
    for name in zf.namelist():
        if name.endswith("/") or not name.lower().endswith((".csv", ".xlsx", ".md")):
            continue
        data = zf.read(name)
        dest = extract_root / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(data)
        print(f"  extracted {name} ({len(data)} bytes)")
        n += 1
    return n


extracted = 0
if file_urls:
    # use the per-file route: pick the largest remote file
    sizes = {}
    for fid, url in file_urls.items():
        with requests.get(url, headers={"Range": "bytes=0-0"}, stream=True,
                          timeout=(30, 120)) as r:
            cr = r.headers.get("Content-Range", "")
            sizes[fid] = int(cr.rsplit("/", 1)[-1]) if "/" in cr else 0
    big_id = max(sizes, key=sizes.get)
    print(f"measurement archive: {big_id} ({sizes[big_id]:,} bytes)")
    with RemoteZip(file_urls[big_id], timeout=(30, 300)) as z:
        print(f"remote archive: {len(z.namelist())} members")
        extracted = extract_small_members(z, PROJECT_DIR)
else:
    # zip-of-zip: locate the inner archive inside the item zip and range-read it
    with RemoteZip(ITEM_ZIP_URL, timeout=(30, 300)) as z:
        infos = z.infolist()
        for i in infos:
            print(f"  member {i.filename}: {i.file_size:,} B method={i.compress_type}")
        inner = [i for i in infos if i.filename.lower().endswith(".zip")]
        assert inner, "no nested archive found"
        zi = max(inner, key=lambda i: i.file_size)
        assert zi.compress_type == 0, (
            "inner zip is deflated; positional ranged reading is not possible - "
            "please retry later, the per-file route may be available again")
        # data offset of the member inside the outer zip (local header is 30 + name + extra)
        with requests.get(ITEM_ZIP_URL,
                          headers={"Range": f"bytes={zi.header_offset}-{zi.header_offset + 29}"},
                          timeout=(30, 120)) as r:
            lh = r.content
        name_len, extra_len = struct.unpack("<HH", lh[26:30])
        data_start = zi.header_offset + 30 + name_len + extra_len
        window = HTTPRangeWindow(ITEM_ZIP_URL, data_start, zi.compress_size)
        with zipfile.ZipFile(window) as inner_zf:
            extracted = extract_small_members(inner_zf, PROJECT_DIR)

assert extracted > 0, "no measurement files were extracted"


## 6. Load the workbook with the authors' `loadMIAPPE` class

Now we instantiate the authors' loader exactly as in their `example_notebook.ipynb`
(cell 1), pointing `data_folder` to the dataset root we just populated. The constructor
reads all seven sheets via `pd.read_excel(..., sheet_name=None)`.

Expected result: the object is created without error (an import proves the dependency
chain `pandas / geopandas / natsort` works).


In [ ]:
import sys
sys.path.insert(0, "/content/repo")

from scripts.data_loader import loadMIAPPE, plot_variable  # authors' code, pinned revision

obj = loadMIAPPE(
    excel_path=xlsx_path,
    data_folder=PROJECT_DIR,
)
print("sheets loaded:", list(obj.excel_file.keys()))


## 7. List the observation units of study `TTADDA_NARO_2021`

This mirrors the paper's Fig. 2: the loader prints the `Observation Unit` rows for the
study — the field (`…_F1`) and its 24 plots (`…_F1P1 … F1P24`, 8 varieties × 3 replicates).

Expected result: a table with 25 rows (1 field + 24 plots) showing `obsUnitId`,
`biologicalMaterialId` (cultivar) and the plot description.


In [ ]:
obs_units = obj.get_observation_unit_info(temp_studyId=study)
print(f"rows returned: {len(obs_units)}")


## 8. Ground coverage of one representative plot over the season

Following the authors' example (`get_observed_variables(..., 'ground_coverage')` +
`plot_variable`), we load the weekly **ground coverage (%)** series of the first plot of
the study, `TTADDA_NARO_2021_F1P1`. The plot shows canopy development and senescence
during the 2021 season at Memuro, Japan.

Expected result: a printed data frame (collectionDate, obsUnitId, ground_coverage in %)
and a line graph over the growing season. *This is the loader's own visualization style,
as demonstrated in the paper's Fig. 2.*


In [ ]:
plot_id = "TTADDA_NARO_2021_F1P1"
gc = obj.get_observed_variables(plot_id, "ground_coverage")[0]
print(gc.to_string(index=False))


The same series is now rendered with the authors' `plot_variable` helper (matplotlib line graph,
collection date on the x axis, ground coverage in % on the y axis).

Expected result: the seasonal canopy-development curve for plot `TTADDA_NARO_2021_F1P1`.

`plot_variable` を用いて同じ系列を折れ線グラフにします（x 軸: 測定日、y 軸: 地被率 %）。


In [ ]:
plot_variable(gc, y_axis="ground_coverage")   # authors' plotting helper


## 9. Ground coverage of all 24 plots (additional visualization)

The paper's Fig. 2 demonstrates a single observation unit; for a compact view of the
variety effect we additionally draw the coverage curves of **all plots of the study** in
one graph (one line per plot, colored by cultivar). This graph is *created for this
notebook* from the actual downloaded data — it is not a published figure.

Expected result: one line graph with 24 curves; cultivars in the legend.


In [ ]:
import matplotlib.pyplot as plt

plots = [u for u in obs_units["obsUnitId"] if u.endswith(tuple(f"P{i}" for i in range(1, 25)))]
fig, ax = plt.subplots(figsize=(11, 5))
for u in plots:
    df_u = obj.get_observed_variables(u, "ground_coverage")[0]
    cultivar = obs_units.loc[obs_units["obsUnitId"] == u, "biologicalMaterialId"].iloc[0]
    df_u = df_u.sort_values("collectionDate")
    ax.plot(df_u["collectionDate"], df_u["ground_coverage"], label=str(cultivar))
ax.set_xlabel("Collection date")
ax.set_ylabel("Ground coverage (%)")
ax.set_title(f"Ground coverage per plot — {study} (created for this notebook)")
ax.grid(alpha=0.3)
ax.legend(fontsize=7, ncol=4)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 10. Harvest yield per plot (`tubwght_total_kgm-2`)

Destructive harvest measurements: total tuber weight per ground area
(`tubwght_total_kgm-2`, kg/m²) per plot, as recorded in the measurement CSVs and indexed
by the MIAPPE workbook. We collect one value per plot into a small table, print it, and
export it as CSV inside the notebook environment.

Expected result: a table with 24 rows (plot, cultivar, yield kg/m²) and a saved
`TTADDA_NARO_2021_yield.csv`.


In [ ]:
import pandas as pd

rows = []
for u in plots:
    cultivar = obs_units.loc[obs_units["obsUnitId"] == u, "biologicalMaterialId"].iloc[0]
    try:
        y = obj.get_observed_variables(u, ["tubwght_total_kgm-2"])[0]
        val = float(y["tubwght_total_kgm-2"].iloc[0])
        date = y["collectionDate"].iloc[0]
    except Exception as e:
        val, date = None, f"missing: {type(e).__name__}"
    rows.append({"obsUnitId": u, "cultivar": cultivar, "collectionDate": date,
                 "tubwght_total_kgm-2 (kg/m2)": val})

yield_df = pd.DataFrame(rows)
yield_df.to_csv("TTADDA_NARO_2021_yield.csv", index=False)
yield_df


### Yield summary statistics and top varieties

A small summary of the harvested yield across the 24 plots (mean / min / max and the
rank of varieties by mean yield). Simple descriptive statistics only — **not** a
statistical analysis as in a hypothesis-testing paper, and based on a single study.

Expected result: a compact `describe()` table plus a per-cultivar mean-yield ranking.


In [ ]:
valid = yield_df.dropna(subset=["tubwght_total_kgm-2 (kg/m2)"])
print(valid["tubwght_total_kgm-2 (kg/m2)"].describe().to_string())
print()
print(valid.groupby("cultivar")["tubwght_total_kgm-2 (kg/m2)"].mean()
      .sort_values(ascending=False).to_string())


## 11. Interpretation and validation record

**What was executed (this notebook, fresh Colab CPU runtime):**

- The authors' pinned loader code (`NPEC-NL/MIAPPE_TTADDA_dataset` @ `2fd6e19`) was used
  unchanged to load the published MIAPPE workbook (MD5-verified).
- The `TTADDA_NARO_2021` measurement CSVs were selectively extracted from the published
  4TU archive with ranged HTTP requests; the ~12.6 GB archive was **not** downloaded.
- Ground-coverage time series and harvest yield per plot were extracted with the
  authors' `get_observed_variables` and visualized with the authors' `plot_variable`
  helper — the same operations demonstrated in the paper's Fig. 2.

**Scope and limitations / スコープと限界**

- This reproduces the paper's *metadata-loading workflow* (a data-descriptor paper has no
  model or accuracy claim to reproduce). It does **not** validate dataset-level science,
  nor the orthomosaic imagery (explicitly out of scope; ~550 GB).
- The workbook links all five studies; only `TTADDA_NARO_2021` is demonstrated here.
- Values shown come from the published v1 data; the notebook prints the actual values, so
  learners can compare their own run against the outputs archived here.

**References**

- Paper: Data in Brief 62:112004 (2025), doi:10.1016/j.dib.2025.112004 (CC BY 4.0 preprint text: PMC12433462)
- Code: https://github.com/NPEC-NL/MIAPPE_TTADDA_dataset (Apache-2.0)
- Data: https://doi.org/10.4121/936b5772-09fc-4856-983d-1f9cc2f38d15 (collection),
  metadata item https://doi.org/10.4121/20ef3f20-ec7b-4236-b13b-e4f7d54a34bf (CC BY-NC-SA 4.0),
  study item https://doi.org/10.4121/f2307c47-9a1a-474a-a0d9-e09ee1b7512c (CC BY-NC-SA 4.0)
